## Fama French Five Factor Benchmark

### Setup

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.size'] = 12

### Configuration

In [ ]:
data_path = Path('../data/Global_Factor_EM.parquet')
results_dir = Path('../results/em/ff_benchmark')
results_dir.mkdir(parents = True, exist_ok = True)

# jkp column names for the five fama-french factor proxies
char_map = {
    'value': 'be_me',
    'profitability': 'ope_be',
    'investment': 'at_gr1',
    'momentum': 'ret_12_1',
    'size': 'me',
}

# fm regression uses the same five proxies only
fm_chars = list(char_map.values())

ret_col = 'ret_exc_lead1m'
rebalance_freq = 6
tc_bps = 25
min_stocks = 30
ret_clip_low = -1.0
ret_clip_high = 1.0

# volatility targeting matching main pipeline config
target_vol = 0.10
vol_lookback = 6
max_leverage_ls = 3.0
max_leverage_lo = 3.0

id_cols = ['id', 'eom', 'excntry', ret_col, 'me']

print(f'data_path: {data_path}')
print(f'results_dir: {results_dir}')

### Load and Process

In [ ]:
schema = pq.read_schema(data_path)
all_col_names = schema.names

factor_cols = list(char_map.values())
fm_available = [c for c in fm_chars if c in all_col_names]
all_chars = list(dict.fromkeys(factor_cols + fm_available))
needed = list(dict.fromkeys([c for c in id_cols + all_chars if c in all_col_names]))

df = pd.read_parquet(data_path, columns = needed)
df['eom'] = pd.to_datetime(df['eom'])
print(f'loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'date range: {df["eom"].min().date()} to {df["eom"].max().date()}')
print(f'countries: {df["excntry"].nunique()}')

for col in all_chars:
    if col in df.columns and df[col].dtype == np.float64:
        df[col] = df[col].astype(np.float32)
if 'me' in df.columns and df['me'].dtype == np.float64:
    df['me'] = df['me'].astype(np.float32)

df[ret_col] = df[ret_col].clip(lower = ret_clip_low, upper = ret_clip_high)
print(f'returns clipped to [{ret_clip_low}, {ret_clip_high}]')


### Build Monthly Cross Sections

In [ ]:
sorted_eoms = sorted(df['eom'].unique())
all_months = {}

for eom in sorted_eoms:
    month = df[df['eom'] == eom].copy()
    month = month[month[ret_col].notna()]
    if len(month) < min_stocks:
        continue

    entry = {
        'ids': month['id'].values,
        'r': month[ret_col].values.astype(np.float64),
        'me': (month['me'].values.astype(np.float64)
               if 'me' in month.columns else np.ones(len(month))),
    }

    for fname, cname in char_map.items():
        entry[fname] = (month[cname].values.astype(np.float64)
                        if cname in month.columns else np.full(len(month), np.nan))

    fm_vals = {}
    for cname in fm_available:
        vals = (month[cname].values.astype(np.float64)
                if cname in month.columns else np.full(len(month), np.nan))
        valid = np.isfinite(vals)
        ranked = np.zeros(len(month))
        if valid.sum() > 5:
            ranked[valid] = pd.Series(vals[valid]).rank(pct = True).values - 0.5
        fm_vals[cname] = ranked

    entry['fm_x'] = np.column_stack([fm_vals[c] for c in fm_available])
    all_months[eom] = entry

sorted_dates = sorted(all_months.keys())
print(f'processed: {len(sorted_dates)} months')
print(f'avg firms/month: {np.mean([len(m["ids"]) for m in all_months.values()]):.0f}')


In [ ]:
def portfolio_metrics(rets):
    rets = np.array(rets, dtype = np.float64)
    if len(rets) == 0:
        return {}
    tw = float((1.0 + rets).prod())
    ann_ret = -1.0 if tw <= 0 else float(tw ** (12.0 / len(rets)) - 1.0)
    ann_vol = float(rets.std() * np.sqrt(12.0))
    sharpe = ann_ret / max(ann_vol, 1e-8)
    se = float(np.sqrt((1.0 + 0.5 * sharpe ** 2) / len(rets)))
    cw = np.cumprod(1.0 + rets)
    pk = np.maximum.accumulate(cw)
    max_dd = float(((pk - cw) / pk).max()) if len(cw) > 0 else 0.0
    return {
        'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe,
        'se_sharpe': se, 'max_dd': max_dd, 'n_obs': len(rets),
    }


def apply_vol_target(monthly_rets, rebalance_indices, target_vol, vol_lookback, max_leverage):
    """
    Apply volatility targeting at rebalance boundaries, matching the main pipeline.

    Estimates annualised volatility from the trailing vol_lookback compounded
    period returns and applies a leverage factor clipped to [1/max_leverage,
    max_leverage] for the next rebalancing window. Active from the vol_lookback-th
    rebalance onward; earlier windows are returned unscaled.
    """
    scaled = np.array(monthly_rets, dtype = np.float64)
    n = len(monthly_rets)
    n_rb = len(rebalance_indices)

    # one compounded period return per rebalancing interval
    period_rets = []
    for i in range(1, n_rb):
        window = np.array(monthly_rets[rebalance_indices[i - 1]:rebalance_indices[i]])
        period_rets.append(float(np.prod(1.0 + window) - 1.0))

    for i in range(n_rb):
        if i < vol_lookback:
            continue
        trailing = np.array(period_rets[max(0, i - vol_lookback):i])
        if len(trailing) < 2:
            continue
        sigma_ann = float(trailing.std() * np.sqrt(12.0 / rebalance_freq))
        lev = float(np.clip(target_vol / max(sigma_ann, 1e-8), 1.0 / max_leverage, max_leverage))
        next_rb = rebalance_indices[i + 1] if i + 1 < n_rb else n
        scaled[rebalance_indices[i]:next_rb] = (
            np.array(monthly_rets[rebalance_indices[i]:next_rb]) * lev
        )

    return scaled



### Market Portfolio

In [ ]:

market_rets = []
for eom in sorted_dates:
    m = all_months[eom]
    valid = np.isfinite(m['me']) & (m['me'] > 0) & np.isfinite(m['r'])
    if valid.sum() < 5:
        continue
    w = m['me'][valid] / m['me'][valid].sum()
    market_rets.append(float((w * m['r'][valid]).sum()))

market_rets = np.array(market_rets)
mkt_m = portfolio_metrics(market_rets)
print(f'\nmarket: {len(market_rets)} months, sharpe: {mkt_m["sharpe"]:.4f}, ann_ret: {mkt_m["ann_ret"] * 100:.2f}%')


### Quintile Factor Portfolios

Equal weighted quintile long short portfolios for each factor. Six month rebalancing with transaction costs.

In [ ]:

def quintile_factor(factor_name, reverse = False):
    rset = set(sorted_dates[::rebalance_freq])
    monthly_rets = []
    rb_indices = []
    li = set()
    si = set()
    pl = set()
    ps = set()
    holdings_log = []

    for eom in sorted_dates:
        m = all_months[eom]
        ids = m['ids']
        r = m['r']
        char_vals = m.get(factor_name)
        if char_vals is None:
            continue

        tcv = 0.0
        if eom in rset:
            rb_indices.append(len(monthly_rets))
            valid = np.isfinite(char_vals)
            if valid.sum() < 10:
                continue
            vi = ids[valid]
            vc = char_vals[valid]
            nq = max(1, int(len(vi) * 0.20))
            so = np.argsort(vc)
            if reverse:
                li = set(vi[so[:nq]].tolist())
                si = set(vi[so[::-1][:nq]].tolist())
            else:
                li = set(vi[so[::-1][:nq]].tolist())
                si = set(vi[so[:nq]].tolist())
            to = (len(li - pl) + len(pl - li) + len(si - ps) + len(ps - si)) / max(nq, 1)
            tcv = to * tc_bps / 10000.0
            pl = li
            ps = si
            holdings_log.append({'eom': str(eom), 'long': sorted(li), 'short': sorted(si)})

        if not li:
            continue
        il = ids.tolist()
        lr = r[np.array([i in li for i in il])]
        sr = r[np.array([i in si for i in il])]
        monthly_rets.append(
            (float(lr.mean()) if len(lr) > 0 else 0.0)
            - (float(sr.mean()) if len(sr) > 0 else 0.0)
            - tcv
        )

    return np.array(monthly_rets), rb_indices, holdings_log


factor_defs = [
    ('value', False), ('momentum', False), ('profitability', False),
    ('investment', True), ('size', True),
]

factor_results = {}
print(f'\nQuintile Factor Portfolios (EM universe, vol-targeted)')
print()
for fname, rev in factor_defs:
    rets, rb_idx, holdings = quintile_factor(fname, reverse = rev)
    if len(rets) == 0:
        print(f'{fname}: no data')
        continue
    rets_scaled = apply_vol_target(rets, rb_idx, target_vol, vol_lookback, max_leverage_ls)
    m_raw = portfolio_metrics(rets)
    m_scaled = portfolio_metrics(rets_scaled)
    factor_results[fname] = {
        'returns_raw': rets, 'returns_scaled': rets_scaled,
        'rb_indices': rb_idx, 'holdings': holdings,
        'metrics_raw': m_raw, 'metrics_scaled': m_scaled,
    }
    print(
        f'{fname:<15} raw sharpe: {m_raw["sharpe"]:.4f}  '
        f'scaled sharpe: {m_scaled["sharpe"]:.4f}  '
        f'ann_ret: {m_scaled["ann_ret"] * 100:.2f}%  '
        f'ann_vol: {m_scaled["ann_vol"] * 100:.2f}%'
    )


### Fama MacBeth Cross Sectional Regression

Each month, regress excess returns on rank normalised characteristics. The time series average of the monthly slope coefficients gives the risk premium estimate. The t statistic tests whether each characteristic commands a significant premium.

In [ ]:

fm_betas = []
fm_dates_used = []

for eom in sorted_dates:
    m = all_months[eom]
    x = m['fm_x']
    r = m['r']
    valid = np.isfinite(r)
    for j in range(x.shape[1]):
        valid = valid & np.isfinite(x[:, j])
    if valid.sum() < len(fm_available) + 5:
        continue
    x_aug = np.column_stack([np.ones(valid.sum()), x[valid]])
    try:
        beta = np.linalg.lstsq(x_aug, r[valid], rcond = None)[0]
        fm_betas.append(beta)
        fm_dates_used.append(eom)
    except np.linalg.LinAlgError:
        continue

fm_betas = np.array(fm_betas)
n_months_fm = len(fm_betas)
fm_mean = fm_betas.mean(axis = 0)
fm_se = fm_betas.std(axis = 0) / np.sqrt(n_months_fm)
fm_tstat = fm_mean / np.maximum(fm_se, 1e-10)

print(f'\nFama-MacBeth Regression ({n_months_fm} months, {len(fm_available)} characteristics)')
print(f'{"variable":<20} {"mean_coef":>10} {"se":>10} {"t_stat":>10} {"p_value":>10}')

coef_names = ['intercept'] + fm_available
fm_results_table = []
for i, name in enumerate(coef_names):
    p_val = 2.0 * (1.0 - stats.t.cdf(abs(fm_tstat[i]), df = n_months_fm - 1))
    sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else ''
    print(f'  {name:<18} {fm_mean[i]:10.5f} {fm_se[i]:10.5f} {fm_tstat[i]:10.5f} {p_val:10.5f} {sig}')
    fm_results_table.append({
        'variable': name, 'mean_coef': float(fm_mean[i]),
        'se': float(fm_se[i]), 't_stat': float(fm_tstat[i]), 'p_value': float(p_val),
    })


### FM Regression Predictive Portfolio

Use the Fama MacBeth regression as a predictive model. Each month, fit the regression on available data and predict next month's returns using an expanding window of coefficient estimates.

In [ ]:

min_history = 60
fm_predictions = {}

for t_idx in range(min_history, len(fm_dates_used)):
    beta_avg = fm_betas[:t_idx].mean(axis = 0)
    pred_date = fm_dates_used[t_idx]
    m = all_months[pred_date]
    pred = beta_avg[0] + m['fm_x'] @ beta_avg[1:]
    valid = np.isfinite(pred)
    if valid.sum() < 10:
        continue
    fm_predictions[pred_date] = {
        'w': pred[valid].astype(np.float32),
        'ids': m['ids'][valid],
        'r': m['r'][valid].astype(np.float32),
    }

print(f'\nFM predictive portfolio: {len(fm_predictions)} out-of-sample months')
if fm_predictions:
    print(f'from {min(fm_predictions.keys()).date()} to {max(fm_predictions.keys()).date()}')

fm_corrs = []
for p in fm_predictions.values():
    if len(p['w']) < 10:
        continue
    c, _ = spearmanr(p['w'], p['r'])
    if not np.isnan(c):
        fm_corrs.append(float(c))
fm_rc = float(np.mean(fm_corrs)) if fm_corrs else 0.0
print(f'FM rank correlation: {fm_rc:.4f}')

keys_fm = sorted(fm_predictions.keys())
rset_fm = set(keys_fm[::rebalance_freq])
fm_q_rets = []
fm_q_rb_indices = []
li = set()
si = set()
pl = set()
ps = set()
fm_q_holdings = []

for eom in keys_fm:
    p = fm_predictions[eom]
    w = p['w']
    ids = p['ids']
    r = p['r']
    tcv = 0.0

    if eom in rset_fm:
        fm_q_rb_indices.append(len(fm_q_rets))
        nq = max(1, int(len(w) * 0.20))
        so = np.argsort(w)
        li = set(ids[so[::-1][:nq]].tolist())
        si = set(ids[so[:nq]].tolist())
        to = (len(li - pl) + len(pl - li) + len(si - ps) + len(ps - si)) / max(nq, 1)
        tcv = to * tc_bps / 10000.0
        pl = li
        ps = si
        fm_q_holdings.append({'eom': str(eom), 'long': sorted(li), 'short': sorted(si)})

    if not li:
        continue
    il = ids.tolist()
    lr = r[np.array([i in li for i in il])]
    sr = r[np.array([i in si for i in il])]
    fm_q_rets.append(
        (float(lr.mean()) if len(lr) > 0 else 0.0)
        - (float(sr.mean()) if len(sr) > 0 else 0.0)
        - tcv
    )

fm_q_rets = np.array(fm_q_rets)
fm_q_rets_scaled = apply_vol_target(fm_q_rets, fm_q_rb_indices, target_vol, vol_lookback, max_leverage_ls)
fm_q_m_raw = portfolio_metrics(fm_q_rets)
fm_q_m = portfolio_metrics(fm_q_rets_scaled)

print(f'FM quintile sharpe (raw): {fm_q_m_raw["sharpe"]:.4f}')
print(
    f'FM quintile sharpe (vol-targeted): {fm_q_m["sharpe"]:.4f}  '
    f'ann_ret: {fm_q_m["ann_ret"] * 100:.2f}%  '
    f'ann_vol: {fm_q_m["ann_vol"] * 100:.2f}%'
)

fm_sw_rets = []
for eom in keys_fm:
    p = fm_predictions[eom]
    w = p['w'].astype(np.float64)
    r = p['r'].astype(np.float64)
    ww = w - w.mean()
    a = np.abs(ww).sum()
    if a > 1e-10:
        fm_sw_rets.append(float((ww / a) @ r))
fm_sw_rets = np.array(fm_sw_rets)
fm_sw_m = portfolio_metrics(fm_sw_rets)
print(f'FM score-weighted sharpe: {fm_sw_m["sharpe"]:.4f}')


In [ ]:
# Evaluate FM predictive portfolio
from scipy.stats import spearmanr

# Rank correlation
fm_corrs = []
for eom, p in fm_predictions.items():
	if len(p['w']) < 10: continue
	c, _ = spearmanr(p['w'], p['r'])
	if not np.isnan(c): fm_corrs.append(c)
fm_rc = float(np.mean(fm_corrs)) if fm_corrs else 0.0
print(f'FM rank correlation: {fm_rc:.4f}')

# Quintile long-short
keys = sorted(fm_predictions.keys())
rset = set(keys[::rebalance_freq])
fm_q_rets = []
li = set()
si = set()
pl = set()
ps = set()
fm_q_holdings = []

for eom in keys:
	p = fm_predictions[eom]
	w = p['w']
	r = p['r']
	ids = p['ids']
	tcv = 0.0
	if eom in rset:
		nq = max(1, int(len(w) * 0.20))
		so = np.argsort(w)
		li = set(ids[so[::-1][:nq]].tolist())
		si = set(ids[so[:nq]].tolist())
		to = (len(li - pl) + len(pl - li) + len(si - ps) + len(ps - si)) / max(nq, 1)
		tcv = to * tc_bps / 10000.0
		pl = li
		ps = si
		fm_q_holdings.append({'eom': str(eom), 'long': sorted(li), 'short': sorted(si)})
	if not li: continue
	il = ids.tolist()
	lr = r[np.array([i in li for i in il])]
	sr = r[np.array([i in si for i in il])]
	fm_q_rets.append((float(lr.mean()) if len(lr) > 0 else 0) - (float(sr.mean()) if len(sr) > 0 else 0) - tcv)

fm_q_rets = np.array(fm_q_rets)
fm_q_m = portfolio_metrics(fm_q_rets)
print(f'FM quintile sharpe: {fm_q_m.get("sharpe", 0):.4f} (se {fm_q_m.get("se_sharpe", 0):.4f})  ret: {fm_q_m.get("ann_ret", 0) * 100:.2f}%')

# Score-weighted
fm_sw_rets = []
for eom in keys:
	p = fm_predictions[eom]
	w = p['w'].astype(np.float64)
	r = p['r'].astype(np.float64)
	ww = w - w.mean()
	a = np.abs(ww).sum()
	if a > 1e-10: fm_sw_rets.append(float((ww / a) @ r))
fm_sw_rets = np.array(fm_sw_rets)
fm_sw_m = portfolio_metrics(fm_sw_rets)
print(f'FM score-wt sharpe: {fm_sw_m.get("sharpe", 0):.4f} (se {fm_sw_m.get("se_sharpe", 0):.4f})  ret: {fm_sw_m.get("ann_ret", 0) * 100:.2f}%')

### Saving Everything

In [ ]:

for fname, fr in factor_results.items():
    np.save(results_dir / f'em_{fname}_returns_raw.npy', fr['returns_raw'])
    np.save(results_dir / f'em_{fname}_returns_scaled.npy', fr['returns_scaled'])
    with open(results_dir / f'em_{fname}_holdings.json', 'w') as fh:
        json.dump(fr['holdings'], fh, indent = 2, default = str)

np.save(results_dir / 'em_market_returns.npy', market_rets)
np.save(results_dir / 'em_fm_quintile_returns_raw.npy', fm_q_rets)
np.save(results_dir / 'em_fm_quintile_returns_scaled.npy', fm_q_rets_scaled)
np.save(results_dir / 'em_fm_scorewt_returns.npy', fm_sw_rets)
with open(results_dir / 'em_fm_holdings.json', 'w') as fh:
    json.dump(fm_q_holdings, fh, indent = 2, default = str)

fm_beta_df = pd.DataFrame(
    fm_betas,
    columns = ['intercept'] + fm_available,
    index = pd.DatetimeIndex(fm_dates_used),
)
fm_beta_df.to_csv(results_dir / 'em_fm_monthly_betas.csv')

summary = {
    'universe': 'EM',
    'fm_characteristics': fm_available,
    'market': mkt_m,
    'factors': {f: fr['metrics_scaled'] for f, fr in factor_results.items()},
    'fm_regression': {
        'n_months': n_months_fm,
        'n_chars': len(fm_available),
        'characteristics': fm_available,
        'coefficients': fm_results_table,
    },
    'fm_portfolio': {
        'quintile_raw': fm_q_m_raw,
        'quintile_scaled': fm_q_m,
        'score_weighted': fm_sw_m,
        'rank_corr': fm_rc,
        'n_oos_months': len(fm_predictions),
    },
}
with open(results_dir / 'em_ff_summary.json', 'w') as fh:
    json.dump(summary, fh, indent = 2, default = float)

print('\nSaved files:')
for fp in sorted(results_dir.glob('em_*')):
    print(f'  {fp.name} ({fp.stat().st_size / 1024:.1f} KB)')



### Summary

In [ ]:

print(f'\nFama-French Benchmark: EM Universe')
print(f'{"strategy":<28} {"sharpe":>8} {"se":>8} {"ann_ret":>9} {"ann_vol":>9} {"max_dd":>8} {"n_obs":>6}')

m = mkt_m
print(
    f'{"market (value-weighted)":<28} {m["sharpe"]:8.4f} {"":8} '
    f'{m["ann_ret"] * 100:8.2f}% {m["ann_vol"] * 100:8.2f}% {"":8} {m["n_obs"]:6d}'
)
for fname in ['value', 'momentum', 'profitability', 'investment', 'size']:
    if fname not in factor_results:
        continue
    m = factor_results[fname]['metrics_scaled']
    print(
        f'{fname:<28} {m["sharpe"]:8.4f} {m["se_sharpe"]:8.4f} '
        f'{m["ann_ret"] * 100:8.2f}% {m["ann_vol"] * 100:8.2f}% '
        f'{m["max_dd"] * 100:7.2f}% {m["n_obs"]:6d}'
    )
m = fm_q_m
print(
    f'{"fm quintile (vol-targeted)":<28} {m["sharpe"]:8.4f} {m["se_sharpe"]:8.4f} '
    f'{m["ann_ret"] * 100:8.2f}% {m["ann_vol"] * 100:8.2f}% '
    f'{m["max_dd"] * 100:7.2f}% {m["n_obs"]:6d}'
)
m = fm_sw_m
print(
    f'{"fm score-weighted":<28} {m["sharpe"]:8.4f} {m["se_sharpe"]:8.4f} '
    f'{m["ann_ret"] * 100:8.2f}% {m["ann_vol"] * 100:8.2f}% '
    f'{m["max_dd"] * 100:7.2f}% {m["n_obs"]:6d}'
)
print(f'\nFM rank correlation: {fm_rc:.4f}')


In [ ]:

fig, axes = plt.subplots(2, 2, figsize = (14, 10))
fig.suptitle('Fama-French Five-Factor Benchmark: EM Universe', fontsize = 14)

all_names = ['Market']
all_sharpes = [mkt_m['sharpe']]
all_ses = [0.0]
for fname in ['value', 'momentum', 'profitability', 'investment', 'size']:
    if fname in factor_results:
        all_names.append(fname.title())
        all_sharpes.append(factor_results[fname]['metrics_scaled']['sharpe'])
        all_ses.append(factor_results[fname]['metrics_scaled']['se_sharpe'])
all_names.extend(['FM Quintile', 'FM Score Wt'])
all_sharpes.extend([fm_q_m['sharpe'], fm_sw_m['sharpe']])
all_ses.extend([fm_q_m['se_sharpe'], fm_sw_m['se_sharpe']])
x = np.arange(len(all_names))
axes[0, 0].bar(x, all_sharpes, yerr = all_ses, capsize = 3)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(all_names, rotation = 35, ha = 'right', fontsize = 8)
axes[0, 0].set_title('Sharpe Ratios (vol-targeted)')
axes[0, 0].grid(axis = 'y', alpha = 0.3)

for fname in ['value', 'momentum', 'profitability', 'investment', 'size']:
    if fname in factor_results:
        axes[0, 1].plot(
            np.cumprod(1.0 + factor_results[fname]['returns_scaled']),
            label = fname.title(),
        )
axes[0, 1].plot(
    np.cumprod(1.0 + fm_q_rets_scaled),
    label = 'FM Quintile', linewidth = 2, linestyle = '--',
)
axes[0, 1].set_title('Cumulative Wealth (vol-targeted)')
axes[0, 1].legend(fontsize = 7)
axes[0, 1].grid(alpha = 0.3)

fm_names = ['intercept'] + fm_available
x2 = np.arange(len(fm_names))
bar_colors = ['steelblue' if abs(fm_tstat[i]) > 1.96 else 'lightgray' for i in range(len(fm_names))]
axes[1, 0].barh(x2, fm_tstat, color = bar_colors)
axes[1, 0].axvline(1.96, color = 'red', linestyle = '--', alpha = 0.5)
axes[1, 0].axvline(-1.96, color = 'red', linestyle = '--', alpha = 0.5)
axes[1, 0].set_yticks(x2)
axes[1, 0].set_yticklabels(fm_names, fontsize = 8)
axes[1, 0].set_title('FM Regression t-statistics')
axes[1, 0].grid(axis = 'x', alpha = 0.3)

axes[1, 1].plot(np.cumprod(1.0 + market_rets), color = 'black', linewidth = 2)
axes[1, 1].set_title('Market Cumulative Wealth (EM, value-weighted)')
axes[1, 1].grid(alpha = 0.3)

plt.tight_layout()
plt.savefig(results_dir / 'em_ff_benchmark.png', dpi = 150, bbox_inches = 'tight')
plt.show()
print(f'plot saved: {results_dir / "em_ff_benchmark.png"}')
